In [1]:
import os,sys
import warnings
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
import anndata as ad
import scanpy as sc
from sklearn.preprocessing import Normalizer
import torch.optim.lr_scheduler as lr_scheduler
from sklearn.preprocessing import StandardScaler
import numpy as np
import torch
import torch.nn as nn
import wandb
import moscot.plotting as mtp
import scipy
from torch.utils.data import DataLoader
from tqdm import tqdm

from moscot import datasets
from moscot.problems.cross_modality import TranslationProblem
from sklearn import preprocessing as pp

from src.samplers.from_dataset import DatasetSampler
from src.samplers.from_loader import PairedLoaderSampler
from torch.utils.data import DataLoader, TensorDataset
from src.samplers.primary import StandardNormalSampler, SwissRollSampler
from src.models.light_gcot import LightGCOT
from src.utils.datasets import (
    get_Splatter_data,
    get_Splatter_dataset,
    get_Splatter_samplers,
)
from sklearn.manifold import TSNE
tsne = TSNE(n_components=2, random_state=50)

#https://moscot.readthedocs.io/en/latest/notebooks/tutorials/600_tutorial_translation.html

env: CUDA_VISIBLE_DEVICES=3


# Data preparation

### PS
1) Source  
$X \in \mathbb{R}^{N \times d_{1}}, N - \text{number of locations}, d_{1} - \text{features dim}$ \
$x = (\mu, \sigma) - \text{for a given location in June}$ \
$N = 1396, d_{1} = 188$ 

2) $Y \in \mathbb{R}^{N \times M \times d_{2}}, N - \text{number of locations}, M - \text{measurements for a given location in January by day}$ \
$M = [1, 31], d_{2} = 94$ 

In [2]:
##########################################
#-------------- RAW DATA -----------------
##########################################

import numpy as np
import pandas as pd

root = 'tabred/data/weather'

data = np.load(f'{root}/X_num.npy')
data = np.stack([d for d in data if sum(np.isnan(d)) == 0])
data_csv = pd.read_csv(f'{root}/csv/X_num.csv')
#train_data = data[train_idx]
#test_data = data[test_idx]

target = np.load(f'{root}/Y.npy')
meta = np.load(f'{root}/X_meta.npy')
meta = np.stack([meta[i] for i, d in enumerate(data) if sum(np.isnan(d)) == 0])
meta_csv = pd.read_csv(f'{root}/csv/X_meta.csv')

names = list(data_csv.columns)
names.append('location')
data_new = np.concatenate((data, meta[:, -2].reshape(-1, 1)), axis=1)

In [3]:
#########################################################
#--------------- Month/location splitted ---------------- 
#########################################################
scaler = StandardScaler()

dict_location_src = {}
for d in data_new:
    if d[-2] == 1.0:
        d_new = d[:-7]
        try:
            dict_location_src[d[-1]].append(d_new)
        except KeyError:
            dict_location_src[d[-1]] = []
            dict_location_src[d[-1]].append(d_new)
     

dict_location_src_new = {}
for key in dict_location_src.keys():
    item = dict_location_src[key]
    item = np.stack(item)
    if item.shape[0] > 1:
        item = (item - np.min(item, axis=0)) / (np.max(item, axis=0) - np.min(item, axis=0) + 1e-1)
        dict_location_src_new[key] = item
dict_location_src = dict_location_src_new
# ------------------------------------------------------------

dict_location_trg = {}
for d in data_new:
    if d[-2] == 6.0:
        d_new = d[:-7]
        try:
            dict_location_trg[d[-1]].append(d_new)
        except KeyError:
            dict_location_trg[d[-1]] = []
            dict_location_trg[d[-1]].append(d_new)
    
dict_location_trg_new = {}
for key in dict_location_trg.keys():
    item = dict_location_trg[key]
    item = np.stack(item)
    if item.shape[0] > 1:
        item = (item - np.min(item, axis=0)) / (np.max(item, axis=0) - np.min(item, axis=0) + 1e-1)
        dict_location_trg_new[key] = item
dict_location_trg = dict_location_trg_new

print(len(dict_location_trg), len(dict_location_src))

1653 1578


In [4]:
#########################################################
#--------------------- X, Y paired ----------------------
#########################################################

chosen_locs = list(dict_location_trg.keys())[:200]
X_pair_orig, Y_pair_orig = [], []
for key in dict_location_src.keys():
    if key not in chosen_locs:
        continue
    item_src = dict_location_src[key] 
    x = np.concatenate([np.mean(item_src, axis=0), np.std(item_src, axis=0)]) # mean, std
    X_pair_orig.append(x)
    item_trg = dict_location_trg[key]
    Y_pair_orig.append(item_trg) # sample
X_pair_orig = np.stack(X_pair_orig)


#########################################################
#----------------------- X, Y ---------------------------
#########################################################

# N x 1 x 2D - src
# N x M x D - trg
 
# sampling: 
# b x 1 x 2D,
# b x M x D -> sample -> b x 1 x D

X_orig = []
for key in dict_location_src.keys():
    if key in chosen_locs:
        continue
    item_src = dict_location_src[key] 
    x = np.concatenate([np.mean(item_src, axis=0), np.std(item_src, axis=0)]) # mean, std
    X_orig.append(x)
X_orig = np.stack(X_orig)

Y_orig = []
for key in dict_location_trg.keys():
    if key in chosen_locs:
        continue
    item_trg = dict_location_trg[key]
    Y_orig.append(item_trg) # sample

In [5]:
print(X_orig.shape, len(Y_orig), X_pair_orig.shape, len(Y_pair_orig))

(1386, 188) 1453 (192, 188) 192


# Running

In [6]:
source_data = X_orig
target_data = Y_orig[0]
X_DIM = source_data.shape[1]
Y_DIM = target_data.shape[1]
#X_DIM = data_set["features"].shape[1]
#Y_DIM = data_set["features"].shape[1]
assert X_DIM > 1
assert Y_DIM > 1

OUTPUT_SEED = 42

N_POTENTIALS = 10
M_POTENTIALS = 1 #10
EPSILON = 1
A_DIAGONAL_INIT = 0.5
L_PAIRED_SAMPLES = len(X_pair_orig)
M_X_UNPAIRED_SAMPLES = 0
N_Y_UNPAIRED_SAMPLES = 0

BATCH_SIZE = 128
SAMPLING_BATCH_SIZE = 128

D_LR = 3e-4  # 1e-3 for eps 0.1, 0.01 and 3e-4 for eps 0.002
D_GRADIENT_MAX_NORM = float("inf")

NUM_LABELED = 10
TRAIN_SUBSET_SIZE = 2

PLOT_EVERY = 1000
MAX_STEPS = 20000
CONTINUE = -1

In [7]:
EXP_COST = "MLP_deep_deep"
EXP_COST_INCLUDED = True
EXP_META_INFO = ""
EXP_NAME = (
    f"Light-GCOT_Batch_Effect_"
    + f"EPSILON_{EPSILON}_"
    + f"N_{N_POTENTIALS}_"
    + f"M_{M_POTENTIALS}_"
    + f"with_{EXP_COST}_"
    + f"cost_included_{EXP_COST_INCLUDED}_"
    + f"N_PAIRED_{NUM_LABELED}_"
    + f"M_UNPAIRED_{len(source_data)}_"
    + EXP_META_INFO
)
OUTPUT_PATH = "../checkpoints/{}".format(EXP_NAME)

config = dict(
    X_DIM=X_DIM,
    Y_DIM=Y_DIM,
    D_LR=D_LR,
    BATCH_SIZE=BATCH_SIZE,
    EPSILON=EPSILON,
    D_GRADIENT_MAX_NORM=D_GRADIENT_MAX_NORM,
    N_POTENTIALS=N_POTENTIALS,
    M_POTENTIALS=M_POTENTIALS,
    A_DIAGONAL_INIT=A_DIAGONAL_INIT,
    N_PAIRED_SAMPLES=NUM_LABELED,
    M_UNPAIRED_SAMPLES=len(source_data),
)

if not os.path.exists(OUTPUT_PATH):
    os.makedirs(OUTPUT_PATH)

In [8]:
#pytorch_total_params = sum(p.numel() for p in D.parameters())
#pytorch_total_params

## Ablation Study

In [9]:
def paired_sampler(X_pair, Y_pair, b_size):
    idxs = np.random.randint(low=0, high=len(X_pair)-1, size=b_size)
    x_pair_batch = torch.tensor(X_pair[idxs]).to('cuda')
    y_pair_batch = np.stack([Y_pair[idx][random.randint(0, len(Y_pair[idx])-1)] for idx in idxs])
    y_pair_batch = torch.tensor(y_pair_batch).to('cuda')
    return x_pair_batch.to(torch.float32), y_pair_batch.to(torch.float32)

def unpaired_sampler(X, Y, b_size):
    # UNPAIRED SAMPLER
    idxs = np.random.randint(low=0, high=len(X)-1, size=b_size)
    idxs_y = np.array([len(X) - idx - 1 for idx in idxs])

    x_batch = torch.tensor(X[idxs]).to('cuda')
    y_batch = np.stack([Y[idx][random.randint(0, len(Y[idx])-1)] for idx in idxs_y])
    y_batch = torch.tensor(y_batch).to('cuda')
    return x_batch.to(torch.float32), y_batch.to(torch.float32)

In [12]:
%load_ext autoreload
%autoreload 2

random.seed(OUTPUT_SEED)
torch.manual_seed(OUTPUT_SEED)
np.random.seed(OUTPUT_SEED)
loader_kwargs = {"num_workers": 0, "pin_memory": True, "generator": torch.Generator(device='cpu')}

def mse(a, b):
    l = (a - b) ** 2
    return l.mean()

#wandb.init(name=EXP_NAME, config=config)
results_df = pd.DataFrame(columns=['L_PAIRED_SAMPLES', 'FOSCTTM_Score'])
MAX_STEPS = 30000
stats = []

# Splitting
L_PAIRED_SAMPLES = 50
L_UNPAIRED_SAMPLES = 50
X, Y = X_orig[:L_UNPAIRED_SAMPLES], Y_orig[-L_UNPAIRED_SAMPLES:]
X_pair, Y_pair = X_pair_orig[:L_PAIRED_SAMPLES], Y_pair_orig[:L_PAIRED_SAMPLES]
X_pair_test, Y_pair_test = X_pair_orig[-100:], Y_pair_orig[-100:]

print(len(X), len(Y), len(X_pair), len(Y_pair), len(X_pair_test), len(Y_pair_test))

for _ in [0]:
    test_size = 100
    print("Training with number of labeled:", L_PAIRED_SAMPLES)
    
    D = LightGCOT(
        x_dim=X_DIM,
        y_dim=Y_DIM,
        n_potentials=10,
        m_potentials=1,
        epsilon=EPSILON,
        sampling_batch_size=SAMPLING_BATCH_SIZE,
        A_diagonal_init=0.4,
        cost_function=EXP_COST,
    )
    D.to('cuda')
    
    D_opt = torch.optim.Adam(D.parameters(), lr=4e-3)
    scheduler = lr_scheduler.StepLR(D_opt, step_size=1000, gamma=0.87)
    
    if CONTINUE > -1:
        D_opt.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, f"D_opt_{CONTINUE}.pt")))
        
    for step in tqdm(range(CONTINUE + 1, MAX_STEPS)):    
        # training loop
        D_opt.zero_grad()
    
        x_batch, y_batch = unpaired_sampler(X, Y, BATCH_SIZE)
        
        log_v_m = D.compute_log_v_m(x_batch)  # [bs x M]
        b_m = D.compute_b_m(x_batch)  # [bs x M x y_dim]
    
        log_w_n = D.compute_log_w_n()
        a_n = D.compute_a_n()  # [N x y_dim]
        A_n = D.compute_A_n()  # [N x y_dim]
    
        f_c = D.compute_dual_potential(log_w_n, a_n, A_n, log_v_m, b_m)
        f = D.compute_primal_potential(y_batch, log_w_n, a_n, A_n)
    
        if EXP_COST_INCLUDED:
            x_pair_batch, y_pair_batch = paired_sampler(X_pair, Y_pair, BATCH_SIZE)
            log_v_m_paired = D.compute_log_v_m(x_pair_batch)  # [bs x M]
            b_m_paired = D.compute_b_m(x_pair_batch)  # [bs x M x y_dim]
    
            c = D.compute_cost(y_pair_batch, log_v_m_paired, b_m_paired)
            D_loss = c.mean() 
            
            if step % 1 == 0:
                D_loss += -(f_c + f).mean()
                
            #print(c.mean().item(), (f_c + f).mean().item())
            D_loss.backward()
        else:
            D_loss = -(f_c + f).mean()
            D_loss.backward()
            
        D_gradient_norm = torch.nn.utils.clip_grad_norm_(D.parameters(), max_norm=D_GRADIENT_MAX_NORM)
        D_opt.step()
        scheduler.step()
    
        if step % 1000 == 0:
            print('D-loss', D_loss.item())
            translated, total_probs = D(torch.tensor(X_pair_test).to('cuda').to(torch.float32),
                                        Y_pair_test)
            print(
                "Average FOSCTTM score of translating ATAC onto RNA: ",
                -total_probs,
            )

    translated, total_probs = D(torch.tensor(X_pair_test).to('cuda').to(torch.float32),
                                Y_pair_test)
    foscttm_score = -total_probs
    
    new_row = pd.DataFrame({
        'L_PAIRED_SAMPLES': [L_PAIRED_SAMPLES],
        'FOSCTTM_Score': [foscttm_score],
    })
    results_df = pd.concat([results_df, new_row], ignore_index=True)
    print('Generated matching')
    #draw_target(X_tsne, T_tsne, traj_plot=20)
    #plt.show()
    print('-------------------------------------------------')
print(results_df)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
50 50 50 50 100 100
Training with number of labeled: 50
1


  0%|                                                 | 0/30000 [00:00<?, ?it/s]

D-loss 188.7578125


  0%|                                        | 13/30000 [00:00<11:35, 43.13it/s]

Average FOSCTTM score of translating ATAC onto RNA:  2764.3993005371094


...

 97%|███████████████████████████████████▊ | 29023/30000 [04:22<00:12, 78.75it/s]

Average FOSCTTM score of translating ATAC onto RNA:  -1295.4631688117981


100%|████████████████████████████████████| 30000/30000 [04:30<00:00, 110.81it/s]


Generated matching
-------------------------------------------------
  L_PAIRED_SAMPLES  FOSCTTM_Score
0               50   -1313.332212


/var/tmp/ipykernel_117208/4077184752.py:103: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results_df = pd.concat([results_df, new_row], ignore_index=True)


In [11]:
pytorch_total_params = sum(p.numel() for p in D.parameters())
pytorch_total_params

4278

In [11]:
# DETERMINISTIC PROBLEM
# seed 42, deterministic

# N_pair | Ours |Ours_deep |  LR   | MLP
# ---------------------------------------
# 100    | 1.71 |  1.20    | 13.25 | 1.11
# 200    | 1.53 |  1.05    | 2.76  | 1.08
# 300    | 1.35 |  1.03    | 1.78  | 1.05
# 400    | 1.24 |  1.01    | 1.55  | 1.06
# 450    | 1.17 |  0.99    | 1.40  | 1.02


# PROBABILISTIC PROBLEM
# - log_p(Y | X)

#  n_pair\n_unpair  |  500  |  100  |   50  |
# -------------------------------------------
#       2           |       |       |       |
#      10           | -1724 |       |       |
#      25           | -1661 | -1427 |       |
#      50           | -1707 | -1446 | -1295 | 
#      90           | -1710 | -1447 | -1273 |

# BASELINE, MLP

#  n_pair           |       | 
# ---------------------------
#       2           |  247  |
#      10           | -56   |
#      25           | -340  |
#      50           | -636  |
#      90           | -676  |

In [ ]:
plt.bar(range(len(stats[9][0].mean(dim=0))), stats[2][0].mean(dim=0))

In [ ]:
plt.bar(range(len(stats[9][0].mean(dim=0))), stats[2][0].mean(dim=0))